# CH3CC2 Computational chemistry with PySCF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChemTeach/blob/main/notebooks/CH3CC2_PySCF_CafChem.ipynb)

## This notebook allows you to:
- Install and load the PySCF, Geometric and RDKit libraries
- Run calculations required for the pc practical

## Requirements:
- This notebook will run faster with more CPU cores (High memoery runtimes have more cores: 8 compared to 2 with the normal CPU runtime)

## Acknowledgements:
- [see the documentation for more details](https://pyscf.org/user/index.html)

## Install and import

In [ ]:
!pip install -q pyscf geometric rdkit

In [ ]:
from pyscf import scf, dft, gto
import pyscf
from pyscf.geomopt.geometric_solver import optimize
from pyscf import lib
from rdkit import Chem
from rdkit.Chem import AllChem
import time, datetime, os
import numpy as np

num_cores = os.cpu_count()
print(f'Number of cores: {num_cores}')

lib.num_threads(num_cores)

## Function for smiles -> PySCF Mol object

In [ ]:
def smiles_to_mol(smiles: str, basis: str = 'sto3g', charge: int = 0, spin:int = 0, symmetry: bool = True):
  '''
  receives a smiles string and returns a pyscf mole object. Adds Hs to
  molecule, optimizes with MMFF by RDKit. Makes and XYZ string,
  and a Mole object.

    Args:
      smiles: SMILES string for molecule
      charge: charge of molecule
      spin: spin multiplicity of molecule

    Returns:
      new_mol: pyscf Mole object
  '''
  atoms_list = ""
  mol = Chem.MolFromSmiles(smiles)
  molH = Chem.AddHs(mol)
  AllChem.EmbedMolecule(molH)
  AllChem.MMFFOptimizeMolecule(molH)
  xyz_string = f""
  for atom in molH.GetAtoms():
    atoms_list += atom.GetSymbol()
    pos = molH.GetConformer().GetAtomPosition(atom.GetIdx())
    xyz_string += f"{atom.GetSymbol()} {pos[0]} {pos[1]} {pos[2]}; "

  new_mol = gto.M(
    atom  = xyz_string,
    charge = charge,
    spin = spin,
    basis = basis,
    symmetry= symmetry
)

  return new_mol

## Electronic structure and total energy of the water molecule in the Hartree-Fock approximation

### Set up water and get the number of primitive and contracted basis functions

In [ ]:
water_mol = gto.M(
    atom = '''O  0.00000  0.00000  0.11779;
              H  0.00000  0.75545 -0.47116;
              H  0.00000 -0.75545 -0.47116''',
    basis = '6-31g*',
    symmetry= True
)

contracted_basis_functions = gto.mole.nao_nr(water_mol) + 1

uncontracted_basis_functions = gto.mole.npgto_nr(water_mol) + 1

print(f'The number of contracted basis functions is: {contracted_basis_functions}')
print(f'The number of uncontracted basis functions is: {uncontracted_basis_functions}')

### perform optimization

In [ ]:
water_hf_631gs = scf.RHF(water_mol)

start = time.time()
finalstructure = optimize(water_hf_631gs)
end = time.time()

### get energy from optimized structure and check structure

In [ ]:
final_water = scf.RHF(finalstructure)
final_energy = final_water.kernel()

print(f'The HF energy is: {final_energy}')
print(f'The HF calculation took {end - start} seconds')
print('The final structure is:')
print(finalstructure.tostring())

In [ ]:
xyz = finalstructure.atom_coords(unit='angs')

centered = [np.subtract(row,xyz[0]) for row in xyz]         # sets O at the origin
bond_length = np.sqrt(sum([x**2 for x in centered[1]]))     # calculates distance from origin to H

for coord1, coord2 in zip(centered[1], centered[2]):        # find which side defines the HH distance
  if coord1 != coord2:
    hh_distance = 2*coord1

theta = np.arcsin(hh_distance*.5/bond_length)               # find half the angle using sine(theta) = opposite / hypotenuse
theta_degrees = theta*180/np.pi                             # change to degrees
bond_angle = 2*(theta_degrees)                              # get full angle

print(f'The bond angle is : {bond_angle}')
print(f'The bond length is: {bond_length} angstroms')

### add optimized coordinates back to original molecule

In [ ]:
water_mol.atom_coords = finalstructure.atom_coords
water_mol.build()
water_mol.atom_coords(unit='angs')

### using optimized coordinates, loop over basis sets top get numbers of basis functions

In [ ]:
for basis in ['sto3g', '3-21g', '6-31g', '6-31g*']:

  print(f'------ Start Analysis for {basis} ---------------------------------')
  water_mol.basis = basis
  water_mol.build()

  contracted_basis_functions = gto.mole.nao_nr(water_mol) + 1
  uncontracted_basis_functions = gto.mole.npgto_nr(water_mol) + 1
  print(f'The number of contracted basis functions in {basis} is: {contracted_basis_functions}')
  print(f'The number of uncontracted basis functions in {basis} is: {uncontracted_basis_functions}')

  water_basis_test = scf.RHF(water_mol)
  energy = water_basis_test.kernel()

  print(f'------------------------------------------------------------------')

### check HF HOMO-LUMO gap

In [ ]:
for i, occ in enumerate(water_basis_test.mo_occ):
  if occ == 0:
    homo_n = i-1
    lumo_n = i
    homo_energy = water_basis_test.mo_energy[homo_n]
    lumo_energy = water_basis_test.mo_energy[lumo_n]
    gap = (water_basis_test.mo_energy[lumo_n] - water_basis_test.mo_energy[homo_n])*27.21
    break

print(f'The HOMO energy is: {homo_energy} ha')
print(f'The LUMO energy is: {lumo_energy} ha')
print(f'The HOMO-LUMO gap is: {gap:.3f} eV')

### Set up new DFT (B3LYP) calculation and optimize

In [ ]:
water_mol = gto.M(
    atom = '''O  0.00000  0.00000  0.11779;
              H  0.00000  0.75545 -0.47116;
              H  0.00000 -0.75545 -0.47116''',
    basis = '6-31g*',
    symmetry= True
)

functional = 'b3lyp'

dft_test = dft.RKS(water_mol)
dft_test.xc = functional
final_dft = optimize(dft_test)

### get optimized DFT energy, move optimized coordinates back to the molecule, and check HOMO-LUMO gap

In [ ]:
water_mol.atom_coords = final_dft.atom_coords
water_mol.build()

dft_test = dft.RKS(water_mol)
dft_test.xc = functional
energy = dft_test.kernel()

for i, occ in enumerate(dft_test.mo_occ):
  if occ == 0:
    homo_n = i-1
    lumo_n = i
    homo_energy = dft_test.mo_energy[homo_n]
    lumo_energy = dft_test.mo_energy[lumo_n]
    gap = (lumo_energy - homo_energy)*27.21
    break

print(f'The HOMO energy is: {homo_energy} ha')
print(f'The LUMO energy is: {lumo_energy} ha')
print(f'The HOMO-LUMO gap is: {gap:.3f} eV')

## Gas Phase

- create molecules from SMILES
- loop over molecules and find:
  * energy
  * number of electrons
  * occupied orbitals
  * HOMO and LUMO energies and gap

### FIRST CELL PERFORMS LOOP: CHECK NEXT CELL FOR RESULTS

In [ ]:
total_out = ''
for smiles in ['C','[H][H]','[C-]#[O+]']:
  new_mol = smiles_to_mol(smiles, basis = '6-31g*', symmetry = False)
  funtional = 'b3lyp'

  dft_test = dft.RKS(new_mol)
  dft_test.xc = functional
  final_dft = optimize(dft_test)

  new_mol.atom_coords = final_dft.atom_coords
  new_mol.build()

  dft_test = dft.RKS(new_mol)
  dft_test.xc = functional
  energy = dft_test.kernel()

  for i, occ in enumerate(dft_test.mo_occ):
    if occ == 0:
      homo_n = i-1
      lumo_n = i
      homo_energy = dft_test.mo_energy[homo_n]
      lumo_energy = dft_test.mo_energy[lumo_n]
      gap = (lumo_energy - homo_energy)*27.21
      break
  out_string = f' -----Data for: {smiles}--------------------------\n'
  out_string += f'Number of electrons: {sum(new_mol.nelec)}\n'
  out_string += f'Number of occupied orbitals: {homo_n+1}\n'
  out_string += f'Energy of the molecule: {energy} ha\n'
  out_string += f'The HOMO energy is: {homo_energy} ha\n'
  out_string += f'The LUMO energy is: {lumo_energy} ha\n'
  out_string += f'The HOMO-LUMO gap is: {gap:.3f} eV\n\n'
  total_out += out_string

### Get results from all molecules

In [ ]:
print(total_out)